In [1]:
import pandas as pd
# set to display all columns
pd.set_option('display.max_columns', None)
# set to print on one line
pd.set_option('display.width', 1000)

In [ ]:
DATASET = "MPDocVQA"
RESULTS_BASE = "."  # directory containing results_w1/, etc.

MODELS = [
    'Phi4', 'Molmo', 'Ovis', 'Llama', 'Llava34', 'Gemma27',
    'Qwen', 'QwenOllama', 'InternVL3', 'InternVL378', 'GPT-4.1', 'O3',
]

# DED normalization weights (documents with 15-25 and >25 tokens are rare; downweighted)
DED_WEIGHTS = {'<15': 1.0, '15-25': 0.16, '>25': 0.336}

# Page indices dropped from PL analysis (too few samples for reliable estimates)
PL_OUTLIER_PAGES = [5, 6, 17, 20]


In [ ]:
def load_csv(path, rows=None):
    """Load a results CSV, filter to MODELS columns, optionally select rows."""
    df = pd.read_csv(f"{RESULTS_BASE}/{path}", index_col=0)[MODELS]
    return df.loc[rows] if rows is not None else df

def save_latex(df, filename):
    df.to_latex(f"table/{filename}", index=True, float_format="%.3f", escape=False)

# AccD - LLM

In [ ]:
df_qur = load_csv("results_w1/results/QUR.csv", rows=["QUR", "QUR_C1", "QUR_C2", "QUR_C3"])
save_latex(df_qur, f"{DATASET}_AccD.tex")
df_qur

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
QUR,0.036946,0.339901,0.216749,0.325123,0.357143,0.394089,0.490148,0.581281,0.241379,0.219212,0.263547,0.162562
QUR_C1,0.044118,0.357843,0.220588,0.313725,0.308824,0.401961,0.500000,0.612745,0.254902,0.274510,0.294118,0.186275
QUR_C2,0.027972,0.328671,0.188811,0.321678,0.440559,0.419580,0.496503,0.538462,0.258741,0.174825,0.258741,0.167832
QUR_C3,0.033898,0.305085,0.271186,0.372881,0.322034,0.305085,0.440678,0.576271,0.152542,0.135593,0.169492,0.067797


In [ ]:
df_ded = load_csv("results_w1/results/QUR_DED.csv")
normalized_ded = df_ded.mul(pd.Series(DED_WEIGHTS), axis=0)
normalized_ded


['<15', '15-25', '>25']

In [ ]:
df_pl = (
    load_csv("results_w1/results/QUR_PL.csv")
    .drop(index=PL_OUTLIER_PAGES)
    .sort_index()
)
df_pl["bin"] = pd.cut(df_pl.index, bins=[0, 4, 8, 100], labels=["<4pg", "4pg-8pg", ">8pg"])
df_pl = df_pl.groupby("bin", observed=False).mean()
df_pl

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
1,0.119048,0.285714,0.309524,0.380952,0.714286,0.452381,0.500000,0.714286,0.190476,0.214286,0.238095,0.166667
2,0.000000,0.283784,0.148649,0.270270,0.500000,0.472973,0.500000,0.567568,0.202703,0.256757,0.202703,0.175676
3,0.050000,0.350000,0.175000,0.375000,0.450000,0.550000,0.525000,0.550000,0.300000,0.300000,0.125000,0.125000
4,0.000000,0.263158,0.236842,0.236842,0.605263,0.578947,0.473684,0.789474,0.315789,0.157895,0.368421,0.236842
5,0.000000,0.257143,0.342857,0.342857,0.171429,0.285714,0.485714,0.571429,0.257143,0.228571,0.200000,0.171429
6,0.057143,0.428571,0.400000,0.428571,0.371429,0.485714,0.542857,0.657143,0.285714,0.200000,0.428571,0.114286
7,0.000000,0.333333,0.250000,0.416667,0.083333,0.583333,0.666667,0.666667,0.250000,0.250000,0.583333,0.416667
8,0.117647,0.470588,0.470588,0.411765,0.235294,0.529412,0.470588,0.529412,0.411765,0.411765,0.352941,0.294118
9,0.000000,0.294118,0.058824,0.352941,0.235294,0.294118,0.470588,0.470588,0.176471,0.235294,0.117647,0.000000
10,0.272727,0.454545,0.272727,0.363636,0.363636,0.272727,0.636364,0.454545,0.454545,0.454545,0.363636,0.272727


In [ ]:
df_nlpe = load_csv("results_w1/results/QUR_NLPE.csv")
df_total = pd.concat([df_qur, normalized_ded, df_pl, df_nlpe])
save_latex(df_total, f"{DATASET}_QUR.tex")
df_total

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
NUMERIC,0.006803,0.340136,0.163265,0.340136,0.312925,0.299320,0.442177,0.564626,0.142857,0.197279,0.197279,0.115646
TEMPORAL,0.148936,0.510638,0.276596,0.553191,0.340426,0.382979,0.638298,0.659574,0.361702,0.255319,0.468085,0.297872
ENTITY,0.019417,0.255663,0.207120,0.297735,0.368932,0.343042,0.420712,0.514563,0.184466,0.145631,0.200647,0.116505
LOCATION,0.038462,0.453846,0.307692,0.346154,0.430769,0.607692,0.684615,0.692308,0.400000,0.300000,0.338462,0.215385
STRUCTURE,0.117647,0.264706,0.176471,0.264706,0.411765,0.264706,0.235294,0.529412,0.176471,0.147059,0.205882,0.088235


# AccP - LLM

In [ ]:
df_ur = load_csv("results_w1/results/UR.csv", rows=["UR", "UR_C1", "UR_C2", "UR_C3"])
save_latex(df_ur, f"{DATASET}_AccP.tex")
df_ur

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
UR,0.211238,0.780228,0.791607,0.795875,0.707681,0.837838,0.881223,0.841520,0.782361,0.818279,0.774538,0.737909
UR_C1,0.224874,0.830402,0.814698,0.844849,0.707286,0.852387,0.901382,0.855200,0.829146,0.849246,0.826633,0.780151
UR_C2,0.188249,0.699041,0.739808,0.724221,0.729017,0.823741,0.850120,0.791391,0.725420,0.756595,0.690647,0.669065
UR_C3,0.204663,0.748705,0.808290,0.748705,0.663212,0.808290,0.865285,0.884868,0.712435,0.823834,0.740933,0.712435


In [ ]:
df_ded  = load_csv("results_w1/results/UR_PAGE_DED.csv")
df_ip   = load_csv("results_w1/results/UR_PAGE_inpage.csv",  rows=["UR_inpage"])
df_op   = load_csv("results_w1/results/UR_PAGE_outpage.csv", rows=["UR_outpage"])
df_nlpe = load_csv("results_w1/results/UR_NLPE.csv")

# df_ur contributes only the UR row (C1/C2/C3 are omitted from the summary table)
df_total = pd.concat([df_ur.loc[["UR"]], df_ded, df_ip, df_op, df_nlpe])
save_latex(df_total, f"{DATASET}_UR.tex")
df_total

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
0,0.231259,0.760962,0.838755,0.771570,0.725601,0.869165,0.881188,0.85110,0.807638,0.863508,0.792786,0.760962
1,0.203229,0.793922,0.769231,0.823362,0.699905,0.807217,0.888889,0.85752,0.775878,0.784425,0.757835,0.714150
>1,0.153623,0.817391,0.666667,0.811594,0.657971,0.802899,0.857971,0.83165,0.698551,0.736232,0.750725,0.715942


# INPAGE - LLM

In [ ]:
df_qp = load_csv("results_w1/results/UR_PAGE_QP.csv")
save_latex(df_qp, f"{DATASET}_AccP_QP.tex")
df_qp

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
TOP_LEFT,0.105036,0.499281,0.506475,0.490647,0.686331,0.663309,0.676259,0.669655,0.410072,0.388489,0.469065,0.415827
TOP_RIGHT,0.297071,0.669456,0.698745,0.719665,0.665272,0.803347,0.845188,0.833773,0.627615,0.627615,0.631799,0.573222
BOTTOM_LEFT,0.246377,0.652174,0.695652,0.672464,0.701449,0.736232,0.811594,0.849026,0.660870,0.689855,0.559420,0.614493
BOTTOM_RIGHT,0.231092,0.789916,0.705882,0.802521,0.714286,0.823529,0.899160,0.844186,0.743697,0.831933,0.689076,0.663866


In [ ]:
df_de = load_csv("results_w1/results/UR_PAGE_DE.csv",
                 rows=["title", "plain text", "figure", "table", "abandon"])
save_latex(df_de, f"{DATASET}_AccP_DED.tex")
df_de

,Phi4,Molmo,Ovis,Llama,Llava34,Gemma27,Qwen,QwenOllama,InternVL3,InternVL378,GPT-4.1,O3
title,0.055556,0.444444,0.388889,0.388889,0.666667,0.583333,0.527778,0.681818,0.333333,0.361111,0.361111,0.305556
plain text,0.208251,0.624754,0.709234,0.646365,0.721022,0.799607,0.823183,0.828221,0.654224,0.722986,0.612967,0.609037
figure,0.176471,0.576471,0.352941,0.647059,0.694118,0.623529,0.800000,0.658333,0.600000,0.611765,0.447059,0.388235
table,0.059829,0.641026,0.529915,0.581197,0.606838,0.675214,0.735043,0.719298,0.350427,0.324786,0.487179,0.461538
abandon,0.433333,0.766667,0.700000,0.800000,0.766667,0.833333,0.800000,0.833333,0.666667,0.866667,0.733333,0.600000
